In [2]:
# instalar las librerias necesarias
pip install sdv sdmetrics

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


# libreria sdv

*Autor: Lara Olivares Martin*

*Fecha: 4 de junio*

In [1]:
# importar librerias

import pandas as pd
import numpy as np
import random

In [2]:
import sdv # generar datos sinteticos
import sdmetrics # evaluar la calidad de los datos generado

from sdv.metadata import SingleTableMetadata
from sdv.single_table import GaussianCopulaSynthesizer

In [3]:
# genera un reporte de los datos generados
from sdmetrics.reports.single_table import QualityReport

In [4]:
# verificar version de SDV
print("SDV:", sdv.__version__)

SDV: 1.37.0


In [5]:
# crear dataset como fuente (el original) para crear los datos sinteticos

dfClientes = pd.DataFrame(

    {
        "cliente_id" : [1,2,3,4,5,6,7,8,9,10],
        "edad" : [23,33,43,28,53,56,43,56,65,40],
        "ingreso_mensual" : [25000,15000,20000,10000,5000,17000,30000,12000,35000,7500],
        "ciudad" : ["Veracruz","Cordoba","Paso del macho","Amatlan","Fortin","Cuitlahuac","Yanga","Cordoba","Orizaba","Cuitlahuac"]
    }
)

In [6]:
# visualizar los primeros registros
dfClientes.head()

,cliente_id,edad,ingreso_mensual,ciudad
0,1,23,25000,Veracruz
1,2,33,15000,Cordoba
2,3,43,20000,Paso del macho
3,4,28,10000,Amatlan
4,5,53,5000,Fortin


In [7]:

# definir los metadatos
metadata = SingleTableMetadata()

In [8]:
# detectar tipos de columnas automaticamente
metadata.detect_from_dataframe(
    data = dfClientes
)

In [9]:
# falla pq : graphviz (dot.exe) no está instalado en el sistema/PATH
# solucion: winget install graphviz (o descargar desde https://graphviz.org/download/ y agregar bin/ al PATH)
metadata.visualize()

C:\Users\tinn\AppData\Roaming\Python\Python311\site-packages\sdv\metadata\visualization.py:131: RuntimeWarning: Graphviz does not seem to be installed on this system. For full metadata visualization capabilities, please make sure to have its binaries propertly installed: https://graphviz.gitlab.io/download/
  warnings.warn(warning_message, RuntimeWarning)


ExecutableNotFound: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH

In [10]:
# como falla el anterior se usa este
metadata.to_dict()

{'primary_key': 'cliente_id',
 'columns': {'cliente_id': {'sdtype': 'id'},
  'edad': {'sdtype': 'numerical'},
  'ingreso_mensual': {'sdtype': 'numerical'},
  'ciudad': {'sdtype': 'categorical'}},
 'METADATA_SPEC_VERSION': 'SINGLE_TABLE_V1'}

In [12]:
# guardar el metadata en json
metadata.save_to_json(
    "dfClientes_metadata.json"
)

In [13]:
# entrenamos el modelo para generar lso datos sinteticos
synthetizer = GaussianCopulaSynthesizer(
    metadata
)

C:\Users\tinn\AppData\Roaming\Python\Python311\site-packages\sdv\single_table\base.py:182: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)


In [14]:
# entrenamiento
synthetizer.fit(
    dfClientes
)

In [15]:
# generamos los datos sinteticos

clientes_sinteticos = synthetizer.sample(
    num_rows=100
)

In [16]:
# visualizar los datos sinteticos generados
clientes_sinteticos.head()

,cliente_id,edad,ingreso_mensual,ciudad
0,16169768,62,24440,Cuitlahuac
1,4918803,33,29322,Fortin
2,1900081,40,17259,Fortin
3,531516,30,18865,Cuitlahuac
4,10211768,49,29881,Yanga


In [17]:
# estadisticas de los datos sinteticos
clientes_sinteticos.describe(include="all")

,cliente_id,edad,ingreso_mensual,ciudad
count,1.000000e+02,100.000000,100.000000,100
unique,NaN,NaN,NaN,8
top,NaN,NaN,NaN,Cordoba
freq,NaN,NaN,NaN,27
mean,8.532170e+06,43.150000,21718.910000,NaN
std,4.687773e+06,12.262594,8952.723575,NaN
min,4.660500e+04,28.000000,6146.000000,NaN
25%,5.008268e+06,31.000000,13661.250000,NaN
50%,8.656048e+06,40.500000,22508.500000,NaN
75%,1.256688e+07,54.250000,29631.250000,NaN


In [18]:
# dataframe contaminado
dfClientesGIGO = clientes_sinteticos.copy()

In [19]:
# colocar edades imposibles
indices = random.sample(list(dfClientesGIGO.index),5)

In [20]:
# contaminar edades del dataframe sintetico
dfClientesGIGO.loc[indices,"edad"] = -5

In [21]:
# agregar registros duplicados
duplicados = dfClientesGIGO.sample(10,random_state=42)
dfClientesGIGO = pd.concat([dfClientesGIGO,duplicados],ignore_index=True)

In [22]:
# verificamos valores duplicados
dfClientesGIGO.duplicated().sum()

np.int64(10)

In [23]:
# estadisticas del dataframe contaminado
dfClientesGIGO.describe(include="all")

,cliente_id,edad,ingreso_mensual,ciudad
count,1.100000e+02,110.000000,110.000000,110
unique,NaN,NaN,NaN,8
top,NaN,NaN,NaN,Cordoba
freq,NaN,NaN,NaN,32
mean,8.614045e+06,40.045455,21104.218182,NaN
std,4.775996e+06,15.511101,9122.475953,NaN
min,4.660500e+04,-5.000000,6146.000000,NaN
25%,4.948624e+06,30.000000,13574.500000,NaN
50%,8.956428e+06,37.500000,22029.000000,NaN
75%,1.274153e+07,53.500000,29474.250000,NaN


In [24]:
# generar el reporte de calidad
report = QualityReport()
report.generate(
    real_data = dfClientes,
    synthetic_data = clientes_sinteticos,
    metadata = metadata.to_dict()
)

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 4/4 [00:00<00:00, 406.91it/s]|
Column Shapes Score: 80.33%

(2/2) Evaluating Column Pair Trends: |██████████| 6/6 [00:00<00:00, 155.88it/s]|
Column Pair Trends Score: 19.5%

Overall Score (Average): 49.92%



In [25]:
# generar el reporte de calidad
report = QualityReport()
report.generate(
    real_data = dfClientes,
    synthetic_data = dfClientesGIGO,
    metadata = metadata.to_dict()
)

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 4/4 [00:00<00:00, 554.45it/s]|
Column Shapes Score: 79.09%

(2/2) Evaluating Column Pair Trends: |██████████| 6/6 [00:00<00:00, 163.89it/s]|
Column Pair Trends Score: 15.45%

Overall Score (Average): 47.27%

